# 技能5 · Day 5 上机：用 LangSmith + tiktoken 生产化营销 Agent

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **LangSmith** `@traceable` 为营销 Agent 配置端到端追踪，查看每步 token/延迟/工具调用
2. 用 **tiktoken** 精确统计 token 成本，结合模型定价计算单次和日均成本
3. 设计**延迟监控**方案：分步计时，识别 P50/P95 瓶颈
4. 实现**灾备降级**（多级 fallback）和 **CI/CD**（pytest 回归测试 + 评估门禁）
5. 用**压测**模拟并发请求，观察延迟/成本/成功率

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：langsmith（LangChain 可观测性平台 SDK）+ tiktoken（OpenAI token 计数）。
营销映射：将营销内容生成 Agent 从 PoC 推向生产环境。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ langsmith 云端 trace 上传需要 `LANGSMITH_API_KEY`（可选，不设则本地追踪不上传）。
> tiktoken 是纯本地库，无需 API key。

In [ ]:
# !pip install langsmith tiktoken -q
# 可选：启用 LangSmith 云端 trace 上传
# export LANGSMITH_TRACING=true
# export LANGSMITH_API_KEY=lsv2_sk_...
# export LANGSMITH_PROJECT=marketing-agent-prod

## 1. 场景背景与营销映射

**生产化对象**：营销内容生成 Agent（生成小红书种草文案/朋友圈广告）。

**PoC 状态**：Agent 能跑通，但：
- 不知道每次请求消耗多少 token / 花多少钱
- 不知道哪个步骤慢（知识库检索 vs LLM 推理）
- 主模型 API 故障时系统直接崩溃
- 代码修改后没有回归测试，可能引入质量问题
- 不知道高并发下系统表现如何

**本上机解决**：用真实生产级工具（langsmith + tiktoken）逐一解决上述问题。

| TODO | 生产化维度 | 工具 | 解决的问题 |
|------|-----------|------|-----------|
| TODO1 | 可观测性 | LangSmith @traceable | 追踪每步调用链 |
| TODO2 | 成本控制 | tiktoken | 精确计 token + 算成本 |
| TODO3 | 延迟优化 | time.perf_counter | 分步计时，识别瓶颈 |
| TODO4 | 灾备降级 | ResilientLLM | 多级 fallback |
| TODO5 | CI/CD | pytest + GitHub Actions | 回归测试 + 评估门禁 |
| TODO6 | 压测 | ThreadPoolExecutor | 并发性能测试 |

**模拟环境说明**：本上机用模拟 LLM 函数（离线可运行），实际使用时替换为真实 OpenAI/Anthropic API 调用。tiktoken 的 token 计数和成本计算是真实的。

In [ ]:
import os
import time
import json
import logging

# ============================================================
# 营销 Agent 模拟环境（离线可运行）
# 实际使用时替换为真实 LLM API 和产品数据库
# ============================================================

# 产品知识库（模拟）
PRODUCT_KB = {
    "烟酰胺精华液": {
        "name": "烟酰胺亮肤精华液",
        "ingredients": "5%烟酰胺",
        "effects": "提亮肤色、收缩毛孔",
        "texture": "清爽水润",
        "price": "199元/30ml",
        "target": "25-35岁女性"
    },
    "丝绒口红": {
        "name": "丝绒哑光口红",
        "ingredients": "哑光质地",
        "effects": "持久不脱色",
        "texture": "丝绒哑光",
        "price": "198元",
        "target": "20-40岁女性"
    },
    "防晒霜": {
        "name": "清透防晒霜",
        "ingredients": "SPF50+ PA++++",
        "effects": "防晒黑",
        "texture": "轻薄透气",
        "price": "159元/50ml",
        "target": "所有肤质"
    },
}

# 营销 Brief 集
MARKETING_BRIEFS = [
    "为烟酰胺精华液写小红书种草文案，目标人群25-35岁女性",
    "为丝绒口红写朋友圈广告",
    "为防晒霜写小红书种草文案",
    "为烟酰胺精华液写朋友圈广告",
    "为防晒霜写朋友圈广告",
]

# 模型定价表（$/token，基于 OpenAI 2026 定价）
MODEL_PRICING = {
    "gpt-4o": {"input": 2.50 / 1_000_000, "output": 10.00 / 1_000_000},
    "gpt-4o-mini": {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000},
}

def mock_search_kb(query: str) -> dict:
    """模拟知识库搜索（实际使用时替换为向量检索）"""
    for key, val in PRODUCT_KB.items():
        if key in query:
            return val
    return {}

def mock_generate_content(brief: str, product_info: dict) -> str:
    """模拟 LLM 生成营销文案（离线可运行）。
    实际使用时替换为：openai.ChatCompletion.create(...) 或 langchain LLM 调用。
    """
    if not product_info:
        return f"抱歉，未找到相关产品信息。Brief: {brief}"
    name = product_info.get("name", "产品")
    effects = product_info.get("effects", "优质")
    price = product_info.get("price", "价格优惠")
    return (
        f"姐妹们！这款{name}真的绝了 "
        f"主打{effects}，效果看得见！"
        f"价格才{price}，性价比拉满！"
        f"评论区扣1获取链接 #{name}"
    )

def mock_llm_primary(prompt: str) -> str:
    """模拟主模型 gpt-4o"""
    product_info = mock_search_kb(prompt)
    return mock_generate_content(prompt, product_info)

def mock_llm_backup(prompt: str) -> str:
    """模拟备用模型 gpt-4o-mini（输出略短）"""
    product_info = mock_search_kb(prompt)
    if not product_info:
        return "未找到产品信息"
    return f"推荐{product_info['name']}，{product_info['effects']}，{product_info['price']}。"

print("环境初始化完成")
print(f"产品知识库: {len(PRODUCT_KB)} 个产品")
print(f"营销Brief: {len(MARKETING_BRIEFS)} 条")
print(f"模型定价: {list(MODEL_PRICING.keys())}")

## TODO 1：配置 LangSmith 追踪，运行营销 Agent

**LangSmith** 是 LangChain 出品的 LLM 可观测性平台。核心 API：
- `@traceable`：装饰器，自动追踪函数调用链（输入/输出/延迟/嵌套）
- `wrap_openai`：包装 OpenAI client，自动记录 LLM 调用
- `Client`：程序化查询 trace 数据（`list_runs`）

即使不配置 `LANGSMITH_API_KEY`，`@traceable` 仍会在本地记录调用链。配置 API key 后可在 https://smith.langchain.com 查看可视化 trace。

In [ ]:
# ===== 你的代码 =====
# TODO: 1) 设置 LangSmith 环境变量（LANGSMITH_TRACING/LANGSMITH_PROJECT）
#       2) 用 @traceable 装饰器定义营销 Agent 函数
#       3) 运行 Agent 并查看 trace
# 提示: from langsmith import traceable, Client
#       @traceable(name="...")  装饰函数
#       Client().list_runs(project_name="...")  查询 trace
trace_result = None  # TODO: 配置追踪并运行 Agent
# ====================

print(f"Agent 输出: {trace_result}")

## TODO 2：用 tiktoken 精确统计 Token 成本

**tiktoken** 是 OpenAI 的 BPE 分词器，比按字符估算精确得多。

**成本计算公式**：
- input_cost = input_tokens × model_pricing["input"]
- output_cost = output_tokens × model_pricing["output"]
- total_cost = input_cost + output_cost

**为什么不能按字符估算**：中文一个字约 1-2 个 token，英文一个单词约 1-2 个 token，标点和 emoji 也消耗 token。只有用 tiktoken 精确计数才能算出真实成本。

In [ ]:
# ===== 你的代码 =====
# TODO: 1) 用 tiktoken 计算 prompt 和 response 的 token 数
#       2) 根据模型定价计算单次成本
#       3) 估算日均成本（10000次/天）
# 提示: import tiktoken
#       enc = tiktoken.encoding_for_model("gpt-4o")
#       tokens = enc.encode(text)
#       cost = num_tokens * MODEL_PRICING[model]["input"]
cost_report = None  # TODO: 计算 token 成本报告
# ====================

print(f"成本报告: {cost_report}")

## 2. 延迟监控：识别 Agent 瓶颈

生产环境中用户期望 5 秒内响应。Agent 的总延迟是各步骤延迟的叠加：
- 知识库检索：通常 < 100ms（向量检索）
- LLM 推理：通常 1-10s（取决于模型和输出长度）
- 工具调用：取决于外部 API 响应时间

**优化策略**：
- 并行化：独立步骤并行执行
- 缓存：相似请求复用响应
- 模型路由：简单任务用小模型（gpt-4o-mini），复杂任务用大模型（gpt-4o）
- 流式输出：LLM 流式返回，用户感知延迟降低

## TODO 3：延迟监控 -- 分步计时，识别瓶颈

为 Agent 各步骤添加 `time.perf_counter()` 计时，多次运行后计算 P50/P95 延迟，识别瓶颈步骤。

In [ ]:
# ===== 你的代码 =====
# TODO: 1) 为 Agent 每个步骤添加计时（search_kb / generate_content）
#       2) 多次运行收集延迟数据
#       3) 计算 P50/P95，识别瓶颈步骤
# 提示: time.perf_counter() 计时
#       运行 20 次，排序取 P50/P95
latency_report = None  # TODO: 延迟监控报告
# ====================

print(f"延迟报告: {latency_report}")

## TODO 4：灾备降级 -- 多级 Fallback

生产环境中 LLM API 可能超时、限流、宕机。系统需要优雅降级：

```
主模型 (gpt-4o)
  ↓ 失败
备用模型 (gpt-4o-mini)
  ↓ 失败
默认模板（预设营销文案模板）
```

**关键设计**：
- 每个模型重试 2 次后切换
- 记录降级日志（哪个模型失败、何时切换）
- 最终降级返回预设模板（而非报错崩溃）

In [ ]:
# ===== 你的代码 =====
# TODO: 1) 实现 ResilientLLM 类，包含模型链 fallback
#       2) 主模型失败时自动切换备用模型
#       3) 所有模型失败时返回降级模板
# 提示: 定义 model_chain = [(name, func), ...]
#       try/except 逐个尝试，失败则 fallback
#       fallback_response() 返回预设模板
resilient_llm = None  # TODO: 实现灾备降级
# ====================

# 测试正常调用和降级
print(f"正常调用结果: {resilient_llm}")

## TODO 5：CI/CD -- 回归测试 + 评估门禁

Agent 系统的 CI/CD 比传统软件更复杂：输出非确定性，不能用精确断言。核心是**评估门禁**：

```
代码提交
  → [1] 代码质量检查（linting/类型检查）
  → [2] 单元测试（Mock LLM，不消耗 token）
  → [3] 集成测试（真实 LLM，检查输出格式/安全/质量）
  → [4] 评估门禁（通过率 >= 90%? 幻觉率 <= 5%? 安全违规 = 0%?）
  → [5] 部署 / 阻止部署
```

本 TODO 用 pytest 风格写回归测试，并模拟 GitHub Actions YAML。

In [ ]:
# ===== 你的代码 =====
# TODO: 1) 用 pytest 风格写 Agent 回归测试函数（至少3个测试）
#       2) 定义评估门禁（通过率 >= 90%）
#       3) 模拟 GitHub Actions YAML 配置
# 提示: def test_xxx(): assert ...
#       门禁: 统计通过率，低于阈值则阻止部署
ci_result = None  # TODO: CI/CD 流水线结果
# ====================

print(f"CI 结果: {ci_result}")

## TODO 6：压测 -- 并发请求性能测试

生产环境可能面临高并发（如营销活动期间）。压测目标：
- 测量吞吐量（req/s）
- 测量 P50/P95 延迟
- 测量总成本
- 检测限流/超时/成功率下降

用 `concurrent.futures.ThreadPoolExecutor` 模拟 N 个并发用户同时请求。

In [ ]:
# ===== 你的代码 =====
# TODO: 1) 用 ThreadPoolExecutor 模拟并发请求（20并发）
#       2) 收集延迟/成本指标
#       3) 计算 P50/P95 延迟和总成本
# 提示: from concurrent.futures import ThreadPoolExecutor, as_completed
#       executor.submit(func, arg)  提交任务
stress_report = None  # TODO: 压测报告
# ====================

print(f"压测报告: {stress_report}")

## 3. 反思与前沿

### 反思问题
1. 你的营销 Agent 在压测中表现出什么瓶颈？（延迟飙升 / 成本超预算 / 限流？）
2. 如果日均 10000 次请求，月成本是多少？是否可接受？如何优化（模型路由/缓存/自建 vLLM）？
3. 灾备降级中，主模型故障时 fallback 到 gpt-4o-mini，输出质量会下降吗？如何监控质量下降？
4. CI 门禁的通过率阈值（90%）是否合理？太高会导致什么问题？太低呢？

### 2026 前沿：推理成本优化
- **vLLM**（https://github.com/vllm-project/vllm）：PagedAttention + 连续批处理，吞吐量 14-24x，自建推理服务替代商业 API
- **投机解码**（arXiv 2211.17192）：小模型生成候选 token，大模型并行验证，延迟降低 2-3x
- **MoE**（arXiv 2401.04088）：Mixture of Experts，总参数大但单次推理计算量小，成本更低
- **LangGraph checkpointer**：Agent 中断恢复，服务重启时从 checkpoint 恢复，节省重复 token 消耗

参考 [vLLM](https://github.com/vllm-project/vllm) + [投机解码论文](https://arxiv.org/abs/2211.17192) + [DeepSeek-MoE](https://arxiv.org/abs/2401.04088)。